# 05 — Train models

Two models:
1. **XGBoost classifier** → P(home_win, draw, away_win)
2. **Poisson regressors** (one per side) → expected goals

Saved under `models/` for the WC2026 prediction notebook.

In [ ]:
import sys, pathlib, joblib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, log_loss, classification_report, mean_absolute_error
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

TT = ROOT / 'data' / 'train_test'
train = pd.read_parquet(TT / 'train.parquet')
test  = pd.read_parquet(TT / 'test.parquet')
FEATURES = pd.read_csv(TT / 'feature_list.csv').feature.tolist()
print('train', train.shape, 'test', test.shape, '| features:', len(FEATURES))


In [ ]:
X_train, y_train = train[FEATURES], train['target']
X_test,  y_test  = test[FEATURES],  test['target']

clf = xgb.XGBClassifier(
    n_estimators=600, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.8,
    objective='multi:softprob', num_class=3,
    eval_metric='mlogloss', tree_method='hist', n_jobs=-1,
)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)
pred = proba.argmax(axis=1)
print('accuracy:', round(accuracy_score(y_test, pred), 4))
print('log loss:', round(log_loss(y_test, proba), 4))
print(classification_report(y_test, pred, target_names=['home','draw','away']))


In [ ]:
# Feature importances
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False)
imp.head(15)


In [ ]:
# --- Poisson regressors for expected goals (scaled — Elo is on a 1500-2200 scale) ---
def poisson_pipe():
    return Pipeline([('scale', StandardScaler()), ('poi', PoissonRegressor(alpha=0.01, max_iter=1000))])
pr_home = poisson_pipe().fit(X_train, train['home_score'])
pr_away = poisson_pipe().fit(X_train, train['away_score'])
pred_h = pr_home.predict(X_test)
pred_a = pr_away.predict(X_test)
print('home goals MAE:', round(mean_absolute_error(test.home_score, pred_h), 3))
print('away goals MAE:', round(mean_absolute_error(test.away_score, pred_a), 3))


In [ ]:
# --- Save models ---
MOD = ROOT / 'models'; MOD.mkdir(parents=True, exist_ok=True)
joblib.dump(clf,     MOD / 'xgb_result.joblib')
joblib.dump(pr_home, MOD / 'poisson_home.joblib')
joblib.dump(pr_away, MOD / 'poisson_away.joblib')
joblib.dump(FEATURES, MOD / 'feature_list.joblib')
print('saved models to', MOD)
